In [38]:
import torch
import torch.nn as nn
import torch.optim as optim
import pytorch_lightning as pl
from sentence_transformers import SentenceTransformer
import numpy as np
from pytorch_lightning.callbacks import Callback
import pandas as pd
import pickle
from pytorch_lightning.callbacks import ModelCheckpoint

In [39]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.get_device_name(0)

'NVIDIA GeForce RTX 3060 Laptop GPU'

In [40]:
items = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/books_data_with_new_id.csv')

full_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/no dup new/amazon_books_ratings_full_filtered_fixed.csv')
train_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/no dup new/amazon_books_ratings_train_filtered_fixed.csv')
val_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/no dup new/amazon_books_ratings_val_filtered_fixed.csv')
test_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/no dup new/amazon_books_ratings_test_filtered_fixed.csv')

In [42]:
items = items.dropna(subset=['item_id'])

duplicated_items = items[items.duplicated(subset='item_id', keep=False)]
duplicated_items

,Title,description,authors,image,previewLink,publisher,publishedDate,infoLink,categories,ratingsCount,item_id


In [45]:
def generate_user_texts_with_history(items, ratings):
    # Initialize user histories with an empty list for each unique user_id
    user_histories = {user_id: [] for user_id in ratings['user_id'].unique()}
    user_texts = []

    # Convert relevant columns to dictionaries for faster access
    items_dict = items.set_index('item_id')[['Title', 'categories', 'authors']].to_dict('index')

    for _, row in ratings.iterrows():
        user_id = row['user_id']
        item_id = row['item_id']
        profile_name = row['profileName']

        # Prepare the user's history (only the last 3 items)
        history_items = []
        for mid in user_histories[user_id][-3:]:
            if mid in items_dict:
                category = items_dict[mid]['categories']

                # Clean and format the category, ensure it is not NaN
                if pd.notna(category):
                    category = category.strip("[]'\"")

                    # Build the history string based on the available data
                    history_entry = f"{category}"

                    if history_entry:
                        history_items.append(history_entry)

        history_str = ", ".join(history_items)

        if history_str:
            combined_features = f"profileName: {profile_name} [SEP] category: {history_str}"
        else:
            combined_features = f"profileName: {profile_name}"

        user_texts.append(combined_features)

        # Update the user history after generating combined features, only if the item_id is valid
        if item_id in items_dict:
            user_histories[user_id].append(item_id)

    return user_texts

In [46]:
train_user_texts = generate_user_texts_with_history(items, train_ratings)
val_user_texts = generate_user_texts_with_history(items, val_ratings)
test_user_texts = generate_user_texts_with_history(items, test_ratings)

['profileName: Ellen C. Falkenberry "ellenf"',
 'profileName: Charles Slovenski',
 'profileName: N. Sausser "pucksau"',
 'profileName: Mary Whipple',
 'profileName: fawls@erols.com',
 'profileName: fawls@erols.com [SEP] category: Fiction',
 'profileName: fawls@erols.com [SEP] category: Fiction, Fiction',
 'profileName: Sai Li',
 'profileName: Sai Li [SEP] category: Fiction',
 'profileName: Sai Li [SEP] category: Fiction, Fiction',
 'profileName: P. Meltzer',
 'profileName: Michael Battaglia',
 'profileName: Michael Battaglia [SEP] category: Fiction',
 'profileName: Gerald Lipsky',
 'profileName: Steve Sailer',
 'profileName: nickoli@rmi.net',
 'profileName: nickoli@rmi.net [SEP] category: Fiction',
 'profileName: Mark Shanks',
 'profileName: Michael Battaglia [SEP] category: Fiction, Fiction',
 'profileName: Travis Cottreau',
 'profileName: Jonathan W. Robie',
 'profileName: Michael Battaglia [SEP] category: Fiction, Fiction, Fiction',
 'profileName: Z23Bull@Aol.com',
 'profileName: la

In [47]:
len(train_user_texts)

262296

In [48]:
def generate_last_user_texts_with_history(items, val_ratings):
    user_histories = {}
    last_user_texts = {}

    # Convert items to a dictionary for faster access
    items_dict = items.set_index('item_id')[['Title', 'categories', 'authors']].to_dict('index')

    # Process the val_ratings
    for _, row in val_ratings.iterrows():
        user_id = row['user_id']
        item_id = row['item_id']
        profile_name = row['profileName']  # Get the profile name directly from the ratings DataFrame

        # Initialize user history if not already done
        if user_id not in user_histories:
            user_histories[user_id] = []

        # Generate the user's history (only the last 3 items)
        history_items = []
        for mid in user_histories[user_id][-3:]:
            if mid in items_dict:
                category = items_dict[mid]['categories']

                # Clean and format the category, ensure it is not NaN
                if pd.notna(category):
                    category = category.strip("[]'\"")

                    # Build the history string based on the available data
                    history_entry = f"{category}"

                    if history_entry:
                        history_items.append(history_entry)

        history_str = ", ".join(history_items)

        # Combine history into the final text format
        if history_str:
            combined_features = f"profileName: {profile_name} [SEP] category: {history_str}"
        else:
            combined_features = f"profileName: {profile_name}"

        # Update the dictionary to keep the last text for each user
        last_user_texts[user_id] = combined_features

        # Update the user history after generating combined features
        if item_id in items_dict:  # Ensure item exists in the dictionary
            user_histories[user_id].append(item_id)

    return last_user_texts

# Generate the last user texts for the validation data
val_last_user_texts = generate_last_user_texts_with_history(items, val_ratings)

In [49]:
val_last_user_texts

{'A3RTKL9KB8KLID': 'profileName: Stan Vernooy [SEP] category: Fiction, Fiction',
 'A3TEH90X39WC8F': 'profileName: Stuart W. Mirsky "swm" [SEP] category: Fiction',
 'A3SOB0CMUBK6XJ': 'profileName: fawls@erols.com',
 'A2QBHNK9H2SVRJ': 'profileName: Angela Linton "Angie" [SEP] category: Fiction',
 'A2YUZKPLUYQDKV': 'profileName: Michael Battaglia [SEP] category: Fiction, Fiction, Fiction',
 'A3927BH5H75LII': 'profileName: Sai Li [SEP] category: Manhattan (New York, N.Y.), Fiction, Philosophy',
 'A2FR8GG77M4TP7': 'profileName: P. Meltzer [SEP] category: Computers',
 'A1GBOCJ943SP8R': 'profileName: Steve Sailer [SEP] category: Fiction',
 'AXOA9OI962P0Q': 'profileName: nickoli@rmi.net [SEP] category: Fiction',
 'ARG7WTYP8L66M': 'profileName: James Paris "Tarnmoor" [SEP] category: Blandings Castle (England : Imaginary place), Catalogs, Union',
 'A2T28ETOO7OBE': 'profileName: Travis Cottreau [SEP] category: Electronic books, Fiction, Ahab, Captain (Fictitious character)',
 'A1UMFRK5YTITOY': 'p

In [53]:
# Combine movie features into a single string for each movie
items['categories'].fillna('', inplace=True)
items['authors'].fillna('', inplace=True)
items['cleaned_categories'] = items['categories'].str.strip('[]').str.replace("'", '')


# Define book_features with the specific condition
items['book_features'] = items.apply(
    lambda row: (
        f"title: {row['Title']} [SEP] category: {row['cleaned_categories']}"
        if row['cleaned_categories'] else
        f"title: {row['Title']}"
    ),
    axis=1
)

items['book_features']

C:\Users\Hooman\AppData\Local\Temp\ipykernel_25096\2261856678.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  items['categories'].fillna('', inplace=True)
C:\Users\Hooman\AppData\Local\Temp\ipykernel_25096\2261856678.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For examp

0         title: Its Only Art If Its Well Hung! [SEP] ca...
1         title: Dr. Seuss: American Icon [SEP] category...
2         title: Wonderful Worship in Smaller Churches [...
3         title: Whispers of the Wicked Saints [SEP] cat...
4         title: Nation Dance: Religion, Identity and Cu...
                                ...                        
212399    title: The Orphan Of Ellis Island (Time Travel...
212400    title: Red Boots for Christmas [SEP] category:...
212401                                         title: Mamaw
212402     title: The Autograph Man [SEP] category: Fiction
212403    title: Student's Solutions Manual for Johnson/...
Name: book_features, Length: 206705, dtype: object

In [54]:
items['cleaned_categories']

0           Comics & Graphic Novels
1         Biography & Autobiography
2                          Religion
3                           Fiction
4                                  
                    ...            
212399             Juvenile Fiction
212400             Juvenile Fiction
212401                             
212402                      Fiction
212403                             
Name: cleaned_categories, Length: 206705, dtype: object

In [56]:
# Create a dictionary for fast lookup
item_features_dict = items.set_index('item_id')['book_features'].to_dict()

# Create lists of user and item texts
item_texts = [item_features_dict[itemId] for itemId in full_ratings['item_id'].unique()]

# Create a mapping from userId and movieId to indices
item_id_to_idx = {itemId: idx for idx, itemId in enumerate(full_ratings['item_id'].unique())}

# Map userId and movieId in ratings_df to indices
train_ratings['item_idx'] = train_ratings['item_id'].map(item_id_to_idx)

# Map userId and movieId in ratings_val to indices
val_ratings['item_idx'] = val_ratings['item_id'].map(item_id_to_idx)

# Map userId and movieId in ratings_test to indices
test_ratings['item_idx'] = test_ratings['item_id'].map(item_id_to_idx)

# Extract user indices, item indices, and ratings
train_item_indices = torch.LongTensor(train_ratings['item_idx'].values).to(device)
train_labels = torch.FloatTensor(train_ratings['rating'].values).to(device)

# Extract user indices, item indices, and ratings for validation
val_item_indices = torch.LongTensor(val_ratings['item_idx'].values).to(device)
val_labels = torch.FloatTensor(val_ratings['rating'].values).to(device)

# Extract user indices, item indices, and ratings for test
test_item_indices = torch.LongTensor(test_ratings['item_idx'].values).to(device)
test_labels = torch.FloatTensor(test_ratings['rating'].values).to(device)


In [57]:
len(test_labels)

36027

In [58]:
test_ratings

,Unnamed: 0.1,Unnamed: 0,item_id,Title,user_id,profileName,rating,timestamp,item_idx
0,36004,2971446,B000G167FA,Silver Pennies,AZUNT3QP2CWTL,"Ellen C. Falkenberry ""ellenf""",5.0,-1,0
1,27466,505724,B0006D9LII,Dance to the Piper,A3VVDE8I22IAJA,Charles Slovenski,5.0,868492800,13
2,21711,743724,B000NKUH32,NATHAN'S RUN.,A3AZ4O4I9S4668,"N. Sausser ""pucksau""",4.0,869270400,15
3,19484,642754,B000Q1RMUE,Eight Months on Ghazzah Street,A319KYEIAZ3SON,Mary Whipple,5.0,869702400,16
4,26662,1962743,B0006BV6RY,Wuthering Heights (College classics in English),A3SOB0CMUBK6XJ,fawls@erols.com,5.0,871084800,28
...,...,...,...,...,...,...,...,...,...
36022,3741,1247106,B000L5XWTA,Of Mice and Men,A1DKMAQDFN0UWA,Erez Davidi,3.0,1362009600,682
36023,28152,818057,B000CPSU9G,The Picture of Dorian Gray,A67VT05EV5EOF,Suzanne L. Goff,5.0,1362009600,1536
36024,14159,2283031,1844560333,Pride and Prejudice,A2GDT5QQSFZD14,Joy Hilda Handley,5.0,1362268800,409
36025,14160,936041,1593355548,Wuthering Heights,A2GDT5QQSFZD14,Joy Hilda Handley,5.0,1362268800,23


In [59]:
from torch.utils.data import Dataset, DataLoader

class CustomTextDataset(Dataset):
    def __init__(self, users, item_ids, ratings):
        self.users = users
        self.item_ids = item_ids
        self.ratings = ratings

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        users = self.users[idx]
        item_id = self.item_ids[idx]
        rating = self.ratings[idx]
        return users, item_id, rating

In [60]:
# Create DataLoader for training data
train_dataset = CustomTextDataset(train_user_texts, train_item_indices, train_labels)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)

# Create DataLoader for validation data
val_dataset = CustomTextDataset(val_user_texts, val_item_indices, val_labels)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=True, drop_last=True)

# Create DataLoader for test data
test_dataset = CustomTextDataset(test_user_texts, test_item_indices, test_labels)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True, drop_last=True)

In [63]:
class TwoTowerModel(pl.LightningModule):
    def __init__(self, user_model_name, item_model_name, embedding_size=768):
        super(TwoTowerModel, self).__init__()
        self.user_model = SentenceTransformer(user_model_name)
        self.item_model = SentenceTransformer(item_model_name)

        self.user_fc = nn.Linear(embedding_size, embedding_size)
        self.item_fc = nn.Linear(embedding_size, embedding_size)

        self.criterion = nn.MSELoss()
        self.epoch_losses = {'train_loss': [], 'val_loss': []}

    def forward(self, user_text, item_text):
        user_embedding = self.user_model.encode(user_text, convert_to_tensor=True).to(device)
        item_embedding = self.item_model.encode(item_text, convert_to_tensor=True).to(device)

        user_output = self.user_fc(user_embedding)
        item_output = self.item_fc(item_embedding)

        dot_product = torch.matmul(user_output.squeeze(), item_output.T)
        dot_product = 4 * torch.sigmoid(dot_product) + 1

        return dot_product

    def training_step(self, batch, batch_idx):
        users, items, ratings = batch

        items = [item_texts[i] for i in items.tolist()]

        preds = self(users, items)

        loss = self.criterion(preds, ratings)
        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        users, items, ratings = batch

        items = [item_texts[i] for i in items.tolist()]

        preds = self(users, items)

        loss = self.criterion(preds, ratings)
        self.log('val_loss', loss)
        return loss

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=1e-5)

class PrintLossesCallback(Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        train_loss = trainer.callback_metrics.get('train_loss')
        if train_loss is not None:
            pl_module.epoch_losses['train_loss'].append(train_loss.item())
            print(f"Epoch {trainer.current_epoch + 1}: Train Loss: {train_loss.item()}")

    def on_validation_epoch_end(self, trainer, pl_module):
        val_loss = trainer.callback_metrics.get('val_loss')
        if val_loss is not None:
            pl_module.epoch_losses['val_loss'].append(val_loss.item())
            print(f"Epoch {trainer.current_epoch + 1}: Val Loss: {val_loss.item()}")

In [64]:
# model = TwoTowerModel(user_model_name='paraphrase-MiniLM-L6-v2', item_model_name='paraphrase-MiniLM-L6-v2')
# model = TwoTowerModel(user_model_name='paraphrase-MiniLM-L12-v2', item_model_name='paraphrase-MiniLM-L12-v2')
model = TwoTowerModel(user_model_name='all-mpnet-base-v2', item_model_name='all-mpnet-base-v2')

# Define the ModelCheckpoint callback
checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',  # Metric to monitor
    dirpath='checkpoints/',  # Directory to save the checkpoints
    filename='with-history-best-checkpoint',  # Filename for the best model
    save_top_k=1,  # Save only the top 1 model
    mode='min'  # Mode to save the best model (min for validation loss)
)

trainer = pl.Trainer(max_epochs=5, log_every_n_steps=1, callbacks=[PrintLossesCallback()], enable_progress_bar=True)
trainer.fit(model, train_dataloader, val_dataloader)

# Print losses after training completes
print("Epoch losses:")
for epoch in range(trainer.max_epochs):
    train_loss = model.epoch_losses['train_loss'][epoch] if epoch < len(model.epoch_losses['train_loss']) else 'N/A'
    val_loss = model.epoch_losses['val_loss'][epoch] if epoch < len(model.epoch_losses['val_loss']) else 'N/A'
    print(f"Epoch {epoch + 1}: Train Loss: {train_loss}, Val Loss: {val_loss}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name       | Type                | Params | Mode 
-----------------------------------------------------------
0 | user_model | SentenceTransformer | 109 M  | train
1 | item_model | SentenceTransformer | 109 M  | train
2 | user_fc    | Linear              | 590 K  | train
3 | item_fc    | Linear              | 590 K  | train
4 | criterion  | MSELoss             | 0      | train
-----------------------------------------------------------
220 M     Trainable params
0         Non-trainable params
220 M     Total params
880.616   Total estimated model params size (MB)


Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

D:\Anaconda\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:475: Your `val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.
D:\Anaconda\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Sanity Checking DataLoader 0:  50%|█████     | 1/2 [00:00<00:00,  4.50it/s]

D:\Anaconda\lib\site-packages\torch\nn\modules\loss.py:535: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 64])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:00<00:00,  4.36it/s]Epoch 1: Val Loss: 2.6961781978607178
                                                                           

D:\Anaconda\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 4098/4098 [25:41<00:00,  2.66it/s, v_num=5]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 510/510 [03:56<00:00,  2.16it/s]Epoch 1: Val Loss: 0.9656608700752258

Epoch 1: 100%|██████████| 4098/4098 [35:49<00:00,  1.91it/s, v_num=5]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 510/510 [04:52<00:00,  1.74it/s]Epoch 2: Val Loss: 0.9606241583824158

Epoch 2: 100%|██████████| 4098/4098 [30:34<00:00,  2.23it/s, v_num=5]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 510/510 [03:34<00:00,  2.37it/s]Epoch 3: Val Loss: 0.9613440632820129

Epoch 3: 100%|██████████| 4098/4098 [30:44<00:00,  2.22it/s, v_num=5]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 510/510 [03:37<00:00,  2.34it/s]Epoch 4: Val Loss: 0.9564009308815002

Epoch 4: 100%|██████████| 4098/4098 [31:09<00:00,  2.19it/s, v_num=5]
Validation: | 

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 4098/4098 [34:58<00:00,  1.95it/s, v_num=5]
Epoch losses:
Epoch 1: Train Loss: 0.9101815819740295, Val Loss: 2.6961781978607178
Epoch 2: Train Loss: 0.8832100033760071, Val Loss: 0.9656608700752258
Epoch 3: Train Loss: 0.7956861257553101, Val Loss: 0.9606241583824158
Epoch 4: Train Loss: 0.9885196685791016, Val Loss: 0.9613440632820129
Epoch 5: Train Loss: 0.6481400728225708, Val Loss: 0.9564009308815002


In [20]:
model.epoch_losses

{'train_loss': [0.9391066431999207,
  1.0028347969055176,
  0.9369568824768066,
  0.9828064441680908,
  0.9409568905830383],
 'val_loss': [10.464865684509277,
  1.1707780361175537,
  1.138741135597229,
  1.120898962020874,
  1.1324385404586792,
  1.116618275642395]}

# Evaluation

In [65]:
# Assuming the training part has been done already, load the best model checkpoint


# best_model_path = './lightning_logs/paraphrase-MiniLM-L12-v2/not-binarized/user_title + store & item_ title + store  _ 5 epoch/checkpoints/epoch=4-step=435.ckpt'
best_model_path = './lightning_logs/version_5/checkpoints/epoch=4-step=20490.ckpt'

# best_model = TwoTowerModel.load_from_checkpoint(best_model_path, user_model_name='paraphrase-MiniLM-L6-v2', item_model_name='paraphrase-MiniLM-L6-v2').to(device)
# best_model = TwoTowerModel.load_from_checkpoint(best_model_path, user_model_name='paraphrase-MiniLM-L12-v2', item_model_name='paraphrase-MiniLM-L12-v2').to(device)
best_model = TwoTowerModel.load_from_checkpoint(best_model_path, user_model_name='all-mpnet-base-v2', item_model_name='all-mpnet-base-v2').to(device)


## Calculations

In [66]:
def get_top_n_items_without_history_unseen_items(model, userId, n):
    # Ensure the model is in evaluation mode
    model.eval()

    # Get the user text for the given userId
    user_text = val_last_user_texts[userId]
    # print(user_text)
    # Encode the user text
    user_embedding = model.user_model.encode(user_text, convert_to_tensor=True).to(device)

    # Compute the scores (dot product between user embedding and each item embedding)
    user_output = model.user_fc(user_embedding).to(device)
    item_output = model.item_fc(full_items_embeddings).to(device)
    dot_product = torch.matmul(user_output, item_output.t()).squeeze()

    # Get items the user has seen in the training and validation data
    seen_items_train = train_ratings[train_ratings['user_id'] == userId]['item_id'].values
    seen_items_val = val_ratings[val_ratings['user_id'] == userId]['item_id'].values
    seen_items = set(np.concatenate((seen_items_train, seen_items_val)))
    # print(dot_product)
    # print(len(dot_product), len(seen_items))
    # Get the top n + len(seen_items) item indices and their scores
    # top_n_scores, top_n_indices = torch.topk(dot_product, n + len(seen_items))
    top_n_scores, top_n_indices = torch.topk(dot_product, n)

    # Map indices back to item IDs
    top_n_item_ids = [list(item_id_to_idx.keys())[list(item_id_to_idx.values()).index(idx.item())] for idx in top_n_indices]
    # print(top_n_item_ids)
    # Filter out seen items
    # unseen_top_n_item_ids = [item for item in top_n_item_ids if item not in seen_items]
    # print(unseen_top_n_item_ids[:n])
    # return unseen_top_n_item_ids[:n]
    # print(top_n_item_ids[:n])
    return top_n_item_ids[:n]


In [67]:
item_texts[:20]

['title: Silver Pennies',
 'title: Playing for the Ashes [SEP] category: Fiction',
 'title: and ladies of the club [SEP] category: Fiction',
 'title: Bethlehem Road [SEP] category: Fiction',
 'title: Hercules, My Shipmate [SEP] category: Fiction',
 'title: HERCULES, MY SHIPMATE [SEP] category: Fiction',
 'title: Guns, Germs, and Steel: The Fates of Human Societies [SEP] category: History',
 'title: The Pathfinder [SEP] category: Fiction',
 'title: The Pathfinder - The Works of J. Fenimore Cooper [SEP] category: United States',
 'title: The Pathfinder, [SEP] category: Fiction',
 "title: The pathfinder (The modern readers' series) [SEP] category: Bumppo, Natty (Fictitious character)",
 'title: Cold Sassy Tree',
 'title: The King Must Die (Cardinal Giant GC-78)',
 'title: Dance to the Piper [SEP] category: Biography & Autobiography',
 'title: Dance to the piper (A Bantam giant)',
 "title: NATHAN'S RUN. [SEP] category: Fiction",
 'title: Eight Months on Ghazzah Street [SEP] category: Ficti

In [68]:
# Assuming full_items_embeddings is already defined
full_items_embeddings = torch.stack([best_model.item_model.encode(item_text, convert_to_tensor=True) for item_text in item_texts]).to(device)

In [69]:
item_texts[8]

'title: The Pathfinder - The Works of J. Fenimore Cooper [SEP] category: United States'

## Type 0

In [70]:
def dcg(scores, k):
    scores = np.asfarray(scores)[:k]
    return np.sum(scores / np.log2(np.arange(2, scores.size + 2)))

def ndcg_at_k(labels, k):
    ideal_labels = sorted(labels, reverse=True)
    return dcg(labels, k) / dcg(ideal_labels, k)

def recall_at_k(labels, relevant_count, k):
    return np.sum(labels[:k]) / relevant_count

def mrr_at_k(labels, k):
    for i, label in enumerate(labels[:k]):
        if label == 1:
            return 1 / (i + 1)
    return 0

def evaluate_user_cf_model(model, test_data, k):
    ndcg_scores = []
    recall_scores = []
    mrr_scores = []

    # Get unique users
    unique_users = test_data['user_id'].unique()

    for user in unique_users:
        # print(user)
        # Get the top N items for the user, filtering out seen items
        recommended_items = get_top_n_items_without_history_unseen_items(model, user, k)
        # recommended_titles = [item_titles.get(item, "Unknown Title") for item in recommended_items]

        # print("Recommended items and their titles:")

        # for item, title in zip(recommended_items, recommended_titles):
        #     print(f"{item}: {title}")
        # recommended_items = ['1844560333', '1593355548', 'B000CPSU9G', '1901768600', 'B000NDSX6C', '158726398X', 'B000P3LVZA', 'B000N6DDJQ', 'B000L5XWTA', 'B0000CO4JZ']

        user_test_data = test_data[test_data['user_id'] == user]
        test_items = user_test_data['item_id'].values

        y_score = [1 if item in test_items else 0 for item in recommended_items]
        # print(y_score)
        ndcg = ndcg_at_k(y_score, k)
        recall = recall_at_k(y_score, len(test_items), k)
        mrr = mrr_at_k(y_score, k)

        ndcg_scores.append(ndcg)
        recall_scores.append(recall)
        mrr_scores.append(mrr)

    # avg_ndcg = np.mean(np.nan_to_num(ndcg_scores, nan=0.0))

    avg_ndcg = np.nanmean(ndcg_scores)
    avg_recall = np.nanmean(recall_scores)
    avg_mrr = np.nanmean(mrr_scores)

    return {
        'NDCG@{}'.format(k): avg_ndcg,
        'Recall@{}'.format(k): avg_recall,
        'MRR@{}'.format(k): avg_mrr,
    }

all_items = items['item_id'].unique()

eval_result = evaluate_user_cf_model(best_model, test_ratings, k=5)
print(eval_result)
eval_result = evaluate_user_cf_model(best_model, test_ratings, k=10)
print(eval_result)

C:\Users\Hooman\AppData\Local\Temp\ipykernel_25096\2329117254.py:7: RuntimeWarning: invalid value encountered in scalar divide
  return dcg(labels, k) / dcg(ideal_labels, k)
C:\Users\Hooman\AppData\Local\Temp\ipykernel_25096\2329117254.py:53: RuntimeWarning: Mean of empty slice
  avg_ndcg = np.nanmean(ndcg_scores)


{'NDCG@5': nan, 'Recall@5': 0.0, 'MRR@5': 0.0}
{'NDCG@10': 0.3286185913860017, 'Recall@10': 0.00011079455523899967, 'MRR@10': 4.39660933488094e-05}


In [ ]:
{'NDCG@5': 0.749980415767912, 'Recall@5': 0.000774772528200495, 'MRR@5': 0.0028648306426084205}

In [235]:
# Find the top 5 most repeated item_id
top_5_item_ids = full_ratings['item_id'].value_counts().head(10)

top_5_item_ids


item_id
1844560333    4079
1593355548    3105
B000CPSU9G    2634
1901768600    1908
B000NDSX6C    1879
158726398X    1843
B000P3LVZA    1644
B000N6DDJQ    1408
B000L5XWTA    1123
B0000CO4JZ    1120
Name: count, dtype: int64

## Type 2

In [71]:
def evaluate_user_cf_model(model, test_data, train_data, val_data, all_items, k):
    ndcg_scores = []

    # Get unique users
    unique_users = test_data['user_id'].unique()

    for user in unique_users:
        # Get the top N items for the user, filtering out seen items
        recommended_items = get_top_n_items_without_history_unseen_items(model, user, k)
        # recommended_items = ['1844560333', '1593355548', 'B000CPSU9G', '1901768600', 'B000NDSX6C']

        # recommended_items = ['1844560333', '1593355548', 'B000CPSU9G', '1901768600', 'B000NDSX6C', '158726398X', 'B000P3LVZA', 'B000N6DDJQ', 'B000L5XWTA', 'B0000CO4JZ']

        user_test_data = test_data[test_data['user_id'] == user]
        test_items = user_test_data['item_id'].values

        y_score = [
            user_test_data[user_test_data['item_id'] == item]['rating'].values[0] if item in test_items else 2.5
            for item in recommended_items
        ]

        ndcg = ndcg_at_k(y_score, k)
        ndcg_scores.append(ndcg)

    avg_ndcg = np.nanmean(ndcg_scores)

    return {
        'NDCG@{}'.format(k): avg_ndcg
    }

all_items = items['item_id'].unique()
# Evaluate the model
eval_result = evaluate_user_cf_model(best_model, test_ratings, train_ratings, val_ratings, all_items, k=5)
print(eval_result)
eval_result = evaluate_user_cf_model(best_model, test_ratings, train_ratings, val_ratings, all_items, k=10)
print(eval_result)

{'NDCG@5': 1.0}
{'NDCG@10': 0.9999616618648126}


## Type 3

In [72]:
def evaluate_user_cf_model(model, test_data, k):
    ndcg_scores = []

    # Get unique users
    unique_users = test_data['user_id'].unique()

    for user in unique_users:
        # Get the top N items for the user, filtering out seen items
        recommended_items = get_top_n_items_without_history_unseen_items(model, user, k)
        # recommended_items = ['1844560333', '1593355548', 'B000CPSU9G', '1901768600', 'B000NDSX6C']
        # recommended_items = ['1844560333', '1593355548', 'B000CPSU9G', '1901768600', 'B000NDSX6C', '158726398X', 'B000P3LVZA', 'B000N6DDJQ', 'B000L5XWTA', 'B0000CO4JZ']

        user_test_data = test_data[test_data['user_id'] == user]
        test_items = user_test_data['item_id'].values
        # print(user)

        y_score = [
            user_test_data[user_test_data['item_id'] == item]['rating'].values[0] if item in test_items else 0
            for item in recommended_items
        ]

        ndcg = ndcg_at_k(y_score, k)
        ndcg_scores.append(ndcg)

    avg_ndcg = np.nanmean(ndcg_scores)

    return {
        'NDCG@{}'.format(k): avg_ndcg
    }

all_items = items['item_id'].unique()
# Evaluate the model
eval_result = evaluate_user_cf_model(best_model, test_ratings, k=5)
print(eval_result)
eval_result = evaluate_user_cf_model(best_model, test_ratings, k=10)
print(eval_result)

C:\Users\Hooman\AppData\Local\Temp\ipykernel_25096\2329117254.py:7: RuntimeWarning: invalid value encountered in scalar divide
  return dcg(labels, k) / dcg(ideal_labels, k)
C:\Users\Hooman\AppData\Local\Temp\ipykernel_25096\138295695.py:25: RuntimeWarning: Mean of empty slice
  avg_ndcg = np.nanmean(ndcg_scores)


{'NDCG@5': nan}
{'NDCG@10': 0.3286185913860017}


## Type 1

In [37]:
def evaluate_user_cf_model(model, test_data, train_data, val_data, all_items, k):
    ndcg_scores = []

    # Get unique users
    unique_users = test_data['user_id'].unique()

    for user in unique_users:
        # Get the top N items for the user, filtering out seen items
        recommended_items = get_top_n_items_without_history_unseen_items(model, user, k)

        user_test_data = test_data[test_data['user_id'] == user]
        test_items = user_test_data['item_id'].values
        # print(user)
        # y_score = [
        #     user_test_data[user_test_data['item_id'] == item]['rating'].values[0] if item in test_items else 0
        #     for item in recommended_items
        # ]
        y_score = [
            1 if (item in test_items and user_test_data[user_test_data['item_id'] == item]['rating'].values[0] == 1) else 0
            for item in recommended_items
        ]
        # y_score = [
        #     1 if (item in test_items and user_test_data[user_test_data['item'] == item]['label'].values[0] == 1) else 0
        #     for item in recommended_items
        # ]

        ndcg = ndcg_at_k(y_score, k)
        ndcg_scores.append(ndcg)

    # avg_ndcg = np.mean(np.nan_to_num(ndcg_scores, nan=0.0))
    avg_ndcg = np.nanmean(ndcg_scores)

    return {
        'NDCG@{}'.format(k): avg_ndcg
    }

all_items = items['item_id'].unique()
# Evaluate the model
eval_result = evaluate_user_cf_model(best_model, test_ratings, train_ratings, val_ratings, all_items, k=5)
print(eval_result)
eval_result = evaluate_user_cf_model(best_model, test_ratings, train_ratings, val_ratings, all_items, k=5)
print(eval_result)

C:\Users\Hooman\AppData\Local\Temp\ipykernel_25096\2329117254.py:7: RuntimeWarning: invalid value encountered in scalar divide
  return dcg(labels, k) / dcg(ideal_labels, k)


{'NDCG@5': 0.5088912804029996}


KeyboardInterrupt: 